In [16]:
# productos.py
# ==============================================================================
# MÓDULO DE INVENTARIO Y PRODUCTOS
# ==============================================================================

class Producto:
    def __init__(self, nombre, precio, stock):
        self.nombre = nombre
        # ENCAPSULAMIENTO: Atributos privados para evitar modificaciones directas
        self.__precio = precio 
        self.__stock = stock   
        
    # GETTERS: Métodos seguros para leer datos privados
    def get_precio(self):
        return self.__precio
        
    def get_stock(self):
        return self.__stock
        
    # SETTERS: Métodos seguros para modificar el stock (Ventas)
    def reducir_stock(self, cantidad):
        if cantidad > 0 and cantidad <= self.__stock:
            self.__stock -= cantidad
            return True
        return False

    # SETTERS: Métodos seguros para modificar el stock (Proveedores)
    def aumentar_stock(self, cantidad):
        if cantidad > 0:
            self.__stock += cantidad
            return True
        return False

# HERENCIA: Bebida hereda de Producto
class Bebida(Producto):
    def __init__(self, nombre, precio, stock, tamano):
        super().__init__(nombre, precio, stock) 
        self.tamano = tamano
        
    # POLIMORFISMO: Impoconsumo del 8% para bebidas preparadas
    def calcular_impuesto(self):
        return self.get_precio() * 0.08 

# HERENCIA: Snack hereda de Producto
class Snack(Producto):
    def __init__(self, nombre, precio, stock, gramos):
        super().__init__(nombre, precio, stock)
        self.gramos = gramos
        
    # POLIMORFISMO: IVA del 19% para snacks procesados
    def calcular_impuesto(self):
        return self.get_precio() * 0.19

In [17]:
# clientes.py
# ==============================================================================
# MÓDULO DE CLIENTES Y DESCUENTOS
# ==============================================================================

class Cliente:
    def __init__(self, nombre, id_cliente):
        self.nombre = nombre
        self.id_cliente = id_cliente

    # Método base que será sobreescrito (Polimorfismo)
    def obtener_descuento(self):
        return 0.0 # 0% de descuento por defecto para clientes genéricos

class Estudiante(Cliente):
    def __init__(self, nombre, id_cliente):
        super().__init__(nombre, id_cliente)
        
    # POLIMORFISMO: 10% de descuento
    def obtener_descuento(self):
        return 0.10 

class Profesor(Cliente):
    def __init__(self, nombre, id_cliente):
        super().__init__(nombre, id_cliente)
        
    # POLIMORFISMO: 5% de descuento
    def obtener_descuento(self):
        return 0.05

In [18]:
# proveedores.py
# ==============================================================================
# MÓDULO DE PROVEEDORES Y ABASTECIMIENTO
# ==============================================================================

class Proveedor:
    def __init__(self, nombre_empresa, nit, ciudad):
        # ENCAPSULAMIENTO: Protegemos los datos sensibles de la empresa
        self.__nombre_empresa = nombre_empresa
        self.__nit = str(nit) 
        self.__ciudad = ciudad

    # GETTER
    def get_nit(self):
        return self.__nit
        
    # SETTER CON VALIDACIÓN (Sanity Check)
    def set_nit(self, nuevo_nit):
        nuevo_nit_limpio = str(nuevo_nit).strip()
        
        if len(nuevo_nit_limpio) == 0:
            print("❌ Error: El NIT no puede estar vacío.")
        elif not nuevo_nit_limpio.isdigit():
            print("❌ Error: El NIT debe contener únicamente números.")
        else:
            self.__nit = nuevo_nit_limpio
            print(f"✅ NIT actualizado correctamente a: {self.__nit}")

    # INTERACCIÓN ENTRE OBJETOS: El proveedor modifica a un objeto 'Producto'
    def suministrar_producto(self, producto, cantidad):
        if cantidad <= 0:
            print("❌ Error: La cantidad a suministrar debe ser mayor a 0.")
            return

        # Llamamos al método seguro del objeto 'producto'
        if producto.aumentar_stock(cantidad):
            print(f"📦 PROVEEDOR: Se han añadido {cantidad} unidades de '{producto.nombre}'. Nuevo stock: {producto.get_stock()}")
        else:
            print("❌ Error interno al actualizar el inventario.")

In [19]:
# ventas.py
# ==============================================================================
# MÓDULO DE VENTAS Y CARRITO DE COMPRAS
# ==============================================================================

class CarritoDeCompras:
    # INTEGRACIÓN: El carrito exige un objeto Cliente al inicializarse
    def __init__(self, cliente):
        self.cliente = cliente 
        self.items =[] 
        self.subtotal = 0.0
        self.total_impuestos = 0.0

    # INTEGRACIÓN: Recibe un objeto Producto
    def agregar_producto(self, producto, cantidad):
        # Intentamos reducir el stock usando el método seguro del producto
        if producto.reducir_stock(cantidad):
            self.items.append({
                "producto": producto, 
                "cantidad": cantidad
            })
            
            # Cálculos financieros leyendo los métodos del objeto
            precio_base = producto.get_precio() * cantidad
            impuesto = producto.calcular_impuesto() * cantidad

            self.subtotal += precio_base
            self.total_impuestos += impuesto
            
            print(f"🛒 VENTAS: Agregado {cantidad}x {producto.nombre} al carrito.")
        else:
            print(f"❌ VENTAS: Stock insuficiente para {producto.nombre}.")

    def generar_factura(self):
        print("\n" + "="*50)
        print("🧾 FACTURA ELECTRÓNICA - CAFETERÍA U. SABANA")
        print(f"👤 Cliente: {self.cliente.nombre} | ID: {self.cliente.id_cliente}")
        print("="*50)
        
        if len(self.items) == 0:
            print("El carrito está vacío.")
        else:
            # Recorremos los objetos guardados
            for item in self.items:
                prod = item["producto"] 
                cant = item["cantidad"]
                total_linea = prod.get_precio() * cant
                print(f"🔸 {prod.nombre.ljust(25)} (x{cant}) : ${total_linea:,.2f}")

            # Lógica financiera
            total_bruto = self.subtotal + self.total_impuestos
            
            # POLIMORFISMO: Python calcula el descuento según el tipo de cliente
            porcentaje_desc = self.cliente.obtener_descuento()
            valor_descuento = total_bruto * porcentaje_desc
            total_pagar = total_bruto - valor_descuento
            
            print("-" * 50)
            print(f"Subtotal:      ${self.subtotal:,.2f} COP")
            print(f"Impuestos:     ${self.total_impuestos:,.2f} COP")
            
            if porcentaje_desc > 0:
                print(f"Descuento ({(porcentaje_desc*100):.0f}%): -${valor_descuento:,.2f} COP")
                
            print(f"TOTAL A PAGAR: ${total_pagar:,.2f} COP")
        print("="*50 + "\n")

In [20]:
# main.py
# ==============================================================================
# ARCHIVO PRINCIPAL (ORQUESTADOR Y BATERÍA DE PRUEBAS)
# ==============================================================================

# 1. IMPORTACIONES: Traemos todas las clases de nuestros módulos

""" 
from productos import Bebida, Snack
from clientes import Estudiante, Profesor
from proveedores import Proveedor
from ventas import CarritoDeCompras 
"""

def main():
    print("☕ INICIANDO BATERÍA DE PRUEBAS 'CAFETERÍA U. SABANA' ☕\n")

    # ==========================================================================
    # PRUEBA 1: MÓDULO DE PRODUCTOS (Herencia y Polimorfismo)
    # ==========================================================================
    print("--- 1. PRUEBAS DE PRODUCTOS ---")
    
    # Instanciamos los objetos
    cafe_tostao = Bebida(nombre="Café de Origen Tostao", precio=5000, stock=50, tamano="Mediano")
    chocolate_jet = Snack(nombre="Chocolatina Jet", precio=1200, stock=100, gramos=12)

    # Probamos lectura de atributos públicos y privados (Getters)
    print(f"Nombre Bebida: {cafe_tostao.nombre}")                # Café de Origen Tostao
    print(f"Precio Bebida: ${cafe_tostao.get_precio():,.0f}")    # 5000 (Viene de atributo privado)
    print(f"Stock Bebida:  {cafe_tostao.get_stock()} unds")      # 50 (Viene de atributo privado)
    print(f"Tamaño Bebida: {cafe_tostao.tamano}")                # Mediano (Atributo propio de Bebida)

    print("\n") # Salto de línea

    print(f"Nombre Snack: {chocolate_jet.nombre}")               # Chocolatina Jet
    print(f"Precio Snack: ${chocolate_jet.get_precio():,.0f}")   # 1200
    print(f"Stock Snack:  {chocolate_jet.get_stock()} unds")     # 100
    print(f"Peso Snack:   {chocolate_jet.gramos}g")              # 12 (Atributo propio de Snack)

    print("\n")

    # Probamos el Polimorfismo (Mismo método, diferente cálculo matemático)
    print(f"Impuesto Café (8%):  ${cafe_tostao.calcular_impuesto():,.1f}")    # 400.0
    print(f"Impuesto Snack (19%): ${chocolate_jet.calcular_impuesto():,.1f}") # 228.0
    print("\n")


    # ==========================================================================
    # PRUEBA 2: MÓDULO DE CLIENTES (Polimorfismo en Descuentos)
    # ==========================================================================
    print("--- 2. PRUEBAS DE CLIENTES ---")
    
    estudiante_ana = Estudiante(nombre="Ana Gómez", id_cliente="1001")
    profesor_carlos = Profesor(nombre="Carlos Ruiz", id_cliente="2002")
    
    print(f"Descuento Estudiante ({estudiante_ana.nombre}): {estudiante_ana.obtener_descuento() * 100}%") # 10.0%
    print(f"Descuento Profesor ({profesor_carlos.nombre}): {profesor_carlos.obtener_descuento() * 100}%") # 5.0%
    print("\n")


    # ==========================================================================
    # PRUEBA 3: MÓDULO DE PROVEEDORES (Encapsulamiento y Validaciones)
    # ==========================================================================
    print("--- 3. PRUEBAS DE PROVEEDORES ---")
    
    proveedor_cafe = Proveedor(nombre_empresa="CoopCafé", nit="900123456", ciudad="Bogotá")
    
    # Probamos el Setter del NIT con errores intencionales (Sanity Check)
    print("\n[Prueba de Seguridad NIT]")
    proveedor_cafe.set_nit("")          # Error: Vacío
    proveedor_cafe.set_nit("900ABC")    # Error: Contiene letras
    proveedor_cafe.set_nit("800987654") # Éxito: Formato válido
    
    # Probamos la interacción entre objetos (Proveedor abastece Producto)
    print("\n[Prueba de Abastecimiento]")
    # Intento fallido (Cantidad 0 o negativa)
    proveedor_cafe.suministrar_producto(producto=cafe_tostao, cantidad=0) 
    # Intento exitoso (Suma 20 al stock actual de 50 -> Queda en 70)
    proveedor_cafe.suministrar_producto(producto=cafe_tostao, cantidad=20) 
    print("\n")


    # ==========================================================================
    # PRUEBA 4: MÓDULO DE VENTAS (Carrito e Integración Total)
    # ==========================================================================
    print("--- 4. PRUEBAS DE CARRITO DE COMPRAS ---")
    
    # IMPORTANTE: En la nueva arquitectura, el Carrito EXIGE un objeto Cliente.
    # Creamos un carrito para la estudiante Ana
    carrito_ana = CarritoDeCompras(cliente=estudiante_ana)

    # Agregamos productos (Caminos de éxito)
    print("\n[Agregando productos válidos]")
    carrito_ana.agregar_producto(cafe_tostao, 2)     # Agrega 2 cafés
    carrito_ana.agregar_producto(chocolate_jet, 3)   # Agrega 3 snacks

    # Agregamos productos (Caminos de error por falta de stock)
    print("\n[Prueba de Seguridad: Exceso de Stock]")
    # El stock del café es 70 (50 iniciales + 20 del proveedor - 2 de la compra anterior = 68 restantes)
    carrito_ana.agregar_producto(cafe_tostao, 100)   # Error: Supera los 68 disponibles
    carrito_ana.agregar_producto(chocolate_jet, 101) # Error: Supera los 97 disponibles

    # Generamos la factura de Ana (Debe aplicar 10% de descuento)
    carrito_ana.generar_factura()


    # ==========================================================================
    # PRUEBA 5: ESCENARIOS ADICIONALES (Profesor y Carrito Vacío)
    # ==========================================================================
    print("--- 5. PRUEBAS ADICIONALES ---")
    
    # Prueba: Factura de un Profesor (Debe aplicar 5% de descuento)
    carrito_carlos = CarritoDeCompras(cliente=profesor_carlos)
    carrito_carlos.agregar_producto(cafe_tostao, 1)
    carrito_carlos.generar_factura()

    # Prueba: Generar factura con carrito vacío (Manejo de errores de UX)
    print("\n[Prueba: Carrito Vacío]")
    carrito_vacio = CarritoDeCompras(cliente=estudiante_ana)
    carrito_vacio.generar_factura()

# Punto de entrada estándar en Python
if __name__ == "__main__":
    main()

☕ INICIANDO BATERÍA DE PRUEBAS 'CAFETERÍA U. SABANA' ☕

--- 1. PRUEBAS DE PRODUCTOS ---
Nombre Bebida: Café de Origen Tostao
Precio Bebida: $5,000
Stock Bebida:  50 unds
Tamaño Bebida: Mediano


Nombre Snack: Chocolatina Jet
Precio Snack: $1,200
Stock Snack:  100 unds
Peso Snack:   12g


Impuesto Café (8%):  $400.0
Impuesto Snack (19%): $228.0


--- 2. PRUEBAS DE CLIENTES ---
Descuento Estudiante (Ana Gómez): 10.0%
Descuento Profesor (Carlos Ruiz): 5.0%


--- 3. PRUEBAS DE PROVEEDORES ---

[Prueba de Seguridad NIT]
❌ Error: El NIT no puede estar vacío.
❌ Error: El NIT debe contener únicamente números.
✅ NIT actualizado correctamente a: 800987654

[Prueba de Abastecimiento]
❌ Error: La cantidad a suministrar debe ser mayor a 0.
📦 PROVEEDOR: Se han añadido 20 unidades de 'Café de Origen Tostao'. Nuevo stock: 70


--- 4. PRUEBAS DE CARRITO DE COMPRAS ---

[Agregando productos válidos]
🛒 VENTAS: Agregado 2x Café de Origen Tostao al carrito.
🛒 VENTAS: Agregado 3x Chocolatina Jet al carrito.